## Lesson 1: Protein Embeddings + Linear Probe
### What you'll learn
- Load a small Protein Language Model (ESM-2, 8M params).
- Run sequences through it to get FIXED-SIZE numerical vectors ("embeddings").
- Train a tiny scikit-learn classifier on top of those embeddings.

### Why this is the right place to start
This is the cheapest way to USE a pLM. The pLM itself is FROZEN — you never
update its weights. You're just treating the pLM as a "feature extractor"
that converts a protein sequence (a string of amino acids) into a vector
that captures evolutionary / structural / functional information.

Then you do classical ML on those vectors: logistic regression, random
forest, whatever. This is called a "linear probe" when the head is linear.

### When this is the right approach
- You have very little labelled data (linear probes resist overfitting).
- You have no GPU.
- You want a quick baseline before going further.

Runs in a few minutes on CPU.

> **Run order matters.** The cells below build on each other. Run them **top to bottom** (Jupyter: *Run → Run All Cells*; VS Code: *Run All*). If you hit `NameError: name 'torch' is not defined` (or similar), you skipped the **Setup** cell — run it first.

## Setup — imports & configuration

**Run this cell first.** It imports every library and defines the module-level constants the rest of the notebook relies on.

In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"
DATASET_NAME = "proteinea/solubility"
N_TRAIN = 500
N_TEST = 200
BATCH_SIZE = 8

In [ ]:
# MLflow tracking -> shared local SQLite store.
# (repo is pip-installed via `pip install -e .`, so this import needs no sys.path shim)
import mlflow_utils as mu
mlflow = mu.setup_mlflow()

### `get_embeddings` (function)

Convert a list of protein sequences into one fixed-size vector each.

Steps inside this function:
1. Tokenize each sequence. ESM-2's tokenizer maps each amino acid to one
   token (plus a special <cls> at the start and <eos> at the end).
2. Run the pLM in `eval` mode and with `torch.no_grad()` (we don't need
   gradients — we're not training).
3. The model returns one vector per token. We MEAN-POOL across tokens to
   get one vector per sequence. We ignore padding tokens during pooling
   (that's what `attention_mask` is for).

In [3]:
def get_embeddings(sequences, model, tokenizer, device, batch_size=BATCH_SIZE):
    all_embeddings = []
    for i in range(0, len(sequences), batch_size):
        batch = sequences[i : i + batch_size]

        # Tokenize the whole batch. Padding=True pads to the longest sequence
        # in this batch. truncation=True caps at max_length to fit in memory.
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        # outputs.last_hidden_state has shape (batch, seq_len, hidden_dim).
        # For ESM-2 8M: hidden_dim == 320.
        hidden = outputs.last_hidden_state

        # Mean-pool over the sequence dimension, ignoring padding tokens.
        # attention_mask is 1 for real tokens, 0 for padding.
        mask = inputs["attention_mask"].unsqueeze(-1).float()  # (B, L, 1)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # (B, hidden_dim)

        all_embeddings.append(pooled.cpu().numpy())

        if (i // batch_size) % 10 == 0:
            print(f"  embedded {min(i + batch_size, len(sequences))}/{len(sequences)}")

    return np.vstack(all_embeddings)

### `main` (function)

In [4]:
def main():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # 1. Load the pre-trained pLM (no fine-tuning).
    # `.eval()` puts dropout etc. into inference mode.
    print(f"Loading model: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

    # 2. Load some labelled protein data.
    print(f"Loading dataset: {DATASET_NAME}")
    ds = load_dataset(DATASET_NAME)
    ds = ds.rename_columns({"sequences": "sequence", "labels": "label"})
    ds = ds.map(lambda b: {"label": [int(x) for x in b["label"]]}, batched=True)
    ds = ds.shuffle(seed=42)
    train = ds["train"].select(range(N_TRAIN))
    test = ds["test"].select(range(N_TEST))

    print(f"Train sequences: {len(train)}, test sequences: {len(test)}")
    print(f"Example: label={train[0]['label']}, seq[:60]={train[0]['sequence'][:60]}...")

    # 3. Convert sequences -> embeddings.
    print("\nExtracting train embeddings...")
    X_train = get_embeddings(train["sequence"], model, tokenizer, device)
    print("Extracting test embeddings...")
    X_test = get_embeddings(test["sequence"], model, tokenizer, device)

    y_train = np.array(train["label"])
    y_test = np.array(test["label"])

    print(f"\nX_train shape: {X_train.shape}  (n_sequences, embedding_dim)")
    print(f"X_test  shape: {X_test.shape}")

    # 4 + 5. Train a classical classifier on top (pLM frozen) and evaluate,
    # logging the whole probe as one MLflow run.
    params = {"model": MODEL_NAME, "dataset": DATASET_NAME,
              "n_train": N_TRAIN, "n_test": N_TEST,
              "head": "LogReg", "C": 1.0, "max_iter": 1000, "pooling": "mean"}
    with mu.run("plm-solubility", "l1_esm2_8M_probe", params=params,
                tags={"lesson": "plm_l1"}):
        print("\nTraining logistic regression on the embeddings...")
        clf = LogisticRegression(max_iter=1000, C=1.0)
        clf.fit(X_train, y_train)

        pred = clf.predict(X_test)
        acc = accuracy_score(y_test, pred)
        f1 = f1_score(y_test, pred)
        baseline = max(y_test.mean(), 1 - y_test.mean())
        mlflow.log_metrics({"test_acc": float(acc), "test_f1": float(f1),
                            "majority_acc": float(baseline)})

    print(f"\nResults:")
    print(f"  Accuracy: {acc:.3f}")
    print(f"  F1:       {f1:.3f}")
    print(f"  Baseline (always-predict-majority): {baseline:.3f}")

    print(
        """
Things to experiment with:
- MODEL_NAME = "facebook/esm2_t12_35M_UR50D" (better, slower)
- Replace LogisticRegression with sklearn.ensemble.RandomForestClassifier
- Try a different pooling strategy:
    pooled = hidden[:, 0, :]              # use only the [CLS] token
    pooled = hidden.max(dim=1).values     # max-pool instead of mean-pool
- Increase N_TRAIN / N_TEST for a more reliable measurement
- Swap DATASET_NAME to "proteinea/solubility" or any other classification set
"""
    )

## Run the lesson

Execute everything above, then run `main()`.

In [5]:
main()

Using device: cuda
Loading model: facebook/esm2_t6_8M_UR50D


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading dataset: proteinea/solubility


Train sequences: 500, test sequences: 200
Example: label=0, seq[:60]=YDTVIRNRQTNDVNYLKRLAKKFGFSETQFMTTYQSSELAANLDEERQVVSRLKINQLPA...

Extracting train embeddings...


...

  embedded 328/500
  embedded 408/500


  embedded 488/500
Extracting test embeddings...
  embedded 8/200


  embedded 88/200
  embedded 168/200



X_train shape: (500, 320)  (n_sequences, embedding_dim)
X_test  shape: (200, 320)



Training logistic regression on the embeddings...

Results:
  Accuracy: 0.535
  F1:       0.384
  Baseline (always-predict-majority): 0.530

Things to experiment with:
- MODEL_NAME = "facebook/esm2_t12_35M_UR50D" (better, slower)
- Replace LogisticRegression with sklearn.ensemble.RandomForestClassifier
- Try a different pooling strategy:
    pooled = hidden[:, 0, :]              # use only the [CLS] token
    pooled = hidden.max(dim=1).values     # max-pool instead of mean-pool
- Increase N_TRAIN / N_TEST for a more reliable measurement
- Swap DATASET_NAME to "proteinea/solubility" or any other classification set



## Comparing models and pooling strategies → see Lesson 5

Want to know which pLM and which pooling strategy give the best probe? That sweep —
`{models} × {mean / max / cls}` on this same frozen-embedding + logistic-regression recipe,
with MLflow runs and a CSV comparison — is its own lesson:

**→ [Lesson 5 — Model & Pooling Comparison](plm_l5_model_comparison.ipynb)**

This lesson stays focused on the single-model embedding + linear-probe workflow.